# Sta-RU Demucs Model Comparison

Take **one** YouTube URL + **one** SRT, render the dub **N times** — one render per Demucs model — and drop all outputs in the same folder with the model name in the filename. Listen back-to-back to pick which model preserves the ambient best for this kind of content.

Everything else (voice, pitch, DD, gain) is held constant across renders so the only variable is the source separator. The video download and the original audio are cached and reused; only the vocals / ambient stems are re-extracted per model.

## 1. GPU check (optional)

In [ ]:
!nvidia-smi | head -20

## 2. Install dependencies

In [ ]:
!apt-get -qq install -y ffmpeg rubberband-cli
!pip install -q edge-tts nest-asyncio yt-dlp srt soundfile numpy scipy demucs deep-translator pyrubberband ipywidgets

## 3. Mount Google Drive (optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Clone the Sta-RU repo

In [ ]:
import os, sys
REPO_DIR = '/content/Sta-RU'
BRANCH = 'claude/compassionate-knuth-M2llL'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 -b $BRANCH https://github.com/lazy-money/sta-ru.git $REPO_DIR
else:
    !cd $REPO_DIR && git pull --quiet
sys.path.insert(0, os.path.join(REPO_DIR, 'colab'))
print('Repo ready at', REPO_DIR, '(branch:', BRANCH, ')')

## 5. The URL to test

Single YouTube URL. Optional `N#:` prefix matches the file-naming convention.

In [ ]:
import ipywidgets as W
from IPython.display import display

url_box = W.Text(
    value='',
    placeholder='https://www.youtube.com/watch?v=...   or   26: https://www.youtube.com/watch?v=...',
    description='URL:',
    layout=W.Layout(width='100%'),
)
display(url_box)

## 6. Upload the SRT

Single file, named `{N#}-{LANG}.srt` (e.g. `26-EN.srt`).

In [ ]:
from google.colab import files
import os, re

SRT_DIR = '/content/srts'
os.makedirs(SRT_DIR, exist_ok=True)
_srts = files.upload()
if len(_srts) != 1:
    print(f'[WARN] expected 1 SRT, got {len(_srts)} — the loop will pick whichever matches the URL N#')
_pattern = re.compile(r'^(\d+)-([A-Z]{2})\.srt$')
for name, content in _srts.items():
    with open(os.path.join(SRT_DIR, name), 'wb') as f:
        f.write(content)
    m = _pattern.match(name)
    if m:
        print(f'  Loaded {name}  (N#{m.group(1)}, lang={m.group(2)})')
    else:
        print(f'  [WARN] {name} does not match N#-LANG.srt — pipeline will skip')

## 7. Options

In [ ]:
import ipywidgets as W
import os
from IPython.display import display
from batch_dub_edge import DEFAULT_VOICES

_drive_mounted = os.path.ismount('/content/drive')
_default_out_path = ('/content/drive/MyDrive/Dubbing/Compare' if _drive_mounted
                     else '/content/output/Compare')

LANG_OPTIONS = list(DEFAULT_VOICES.keys())

lang_w           = W.Dropdown(options=LANG_OPTIONS, value='EN', description='Target lang:')
gender_w         = W.RadioButtons(options=[('Male', 'M'), ('Female', 'F')], value='M', description='Voice gender:')
voice_custom_w   = W.Text(value='', placeholder='Optional: override the default voice',
                          description='Custom voice:', layout=W.Layout(width='600px'))
pitch_w          = W.IntSlider(value=-5, min=-20, max=20, step=1, description='Pitch (Hz):')
dynamic_dur_w    = W.Checkbox(value=True, description='Dynamic duration')
skip_silent_w    = W.Checkbox(value=True, description='Skip TTS where original is silent')
translate_title_w= W.Checkbox(value=False, description='Translate title (off by default — same name across runs)')

models_w = W.Text(
    value='htdemucs, htdemucs_ft, hdemucs_mmi, mdx_extra',
    description='Models:',
    layout=W.Layout(width='100%'),
)
segment_w = W.IntText(value=10, description='Segment (s):',
                      tooltip='Chunk size for Demucs. 10s keeps RAM low; set to 0 for model default.')

out_path_w   = W.Text(value=_default_out_path, description='Output dir:', layout=W.Layout(width='600px'))
cache_path_w = W.Text(value='/tmp/sta-ru-cache', description='Cache dir:', layout=W.Layout(width='600px'))

display(lang_w, gender_w, voice_custom_w, pitch_w,
        dynamic_dur_w, skip_silent_w, translate_title_w,
        models_w, segment_w, out_path_w, cache_path_w)
print()
print('Models: comma-separated list. Suggested set covers the main architectures:')
print('  htdemucs       — default (hybrid transformer, trained on music)')
print('  htdemucs_ft    — fine-tuned variant of htdemucs, often gentler on non-music')
print('  hdemucs_mmi    — older hybrid, lighter on RAM, less aggressive')
print('  mdx_extra      — different architecture (MDX), worth trying for non-music ambient')
print('Segment: 10s = safe for hour-long videos on Colab; 0 lets each model use its default.')
if not _drive_mounted:
    print()
    print('NOTE: Drive not mounted — output defaulted to local. Mount Drive and re-run this cell if needed.')

In [ ]:
# Lock in configuration
import os

MODELS = [m.strip() for m in models_w.value.split(',') if m.strip()]
if not MODELS:
    raise RuntimeError('No models specified.')

CONFIG = {
    'lang':                 lang_w.value,
    'gender':               gender_w.value,
    'voice':                voice_custom_w.value.strip() or None,
    'pitch_st':             pitch_w.value,
    'dynamic_duration':     dynamic_dur_w.value,
    'skip_silent_segments': skip_silent_w.value,
    'translate_titles':     translate_title_w.value,
    'output_dir':           out_path_w.value,
    'srt_dir':              SRT_DIR,
    'cache_root':           cache_path_w.value or None,
    'demucs_segment':       segment_w.value if segment_w.value > 0 else None,
    'models':               MODELS,
}

url_text = (url_box.value or '').strip()
if not url_text:
    raise RuntimeError('No URL provided.')
URL_SOURCE = [url_text]

from batch_dub_edge import resolve_voice
effective_voice = resolve_voice(CONFIG['lang'], CONFIG['gender'], CONFIG['voice'])
print(f'URL:             {url_text}')
print(f'Effective voice: {effective_voice}')
print(f'Models to test:  {MODELS}')
print()
print('Configuration:')
for k, v in CONFIG.items():
    print(f'  {k:22} = {v}')

if CONFIG['output_dir'].startswith('/content/drive/') and not os.path.ismount('/content/drive'):
    _fallback = '/content/output/Compare'
    print()
    print(f'  [NOTICE] Drive is not mounted but output_dir points there.')
    print(f'           Falling back to local: {_fallback}')
    CONFIG['output_dir'] = _fallback

## 8. Run the comparison

For each model: wipes the cached `ambient.wav` / `vocals.wav` so Demucs re-runs, renders the dub, then renames the output to include the model name. The video download and the raw audio are reused across iterations to save time.

In [ ]:
import shutil, time
from pathlib import Path
from batch_dub_edge import run_batch

cache_root = Path(CONFIG['cache_root']) if CONFIG['cache_root'] else None
out_dir = Path(CONFIG['output_dir'])
out_dir.mkdir(parents=True, exist_ok=True)

results_per_model = {}
t0_total = time.time()
for i, model in enumerate(CONFIG['models'], 1):
    print(f'\n{"="*60}\n[{i}/{len(CONFIG["models"])}]  MODEL: {model}\n{"="*60}')

    # Clear cached stems so Demucs runs fresh for this model.
    # Video.mp4 and orig.wav stay cached.
    if cache_root and cache_root.exists():
        for url_dir in cache_root.iterdir():
            if not url_dir.is_dir():
                continue
            for fname in ('ambient.wav', 'vocals.wav'):
                (url_dir / fname).unlink(missing_ok=True)

    t0 = time.time()
    try:
        res = run_batch(
            urls=URL_SOURCE,
            srt_dir=CONFIG['srt_dir'],
            output_dir=str(out_dir),
            lang=CONFIG['lang'],
            gender=CONFIG['gender'],
            voice=CONFIG['voice'],
            pitch_st=CONFIG['pitch_st'],
            translate_titles=CONFIG['translate_titles'],
            remove_voice=True,
            dynamic_duration=CONFIG['dynamic_duration'],
            skip_silent_segments=CONFIG['skip_silent_segments'],
            cache_root=str(cache_root) if cache_root else None,
            range_expr='all',
            demucs_model=model,
            demucs_segment=CONFIG['demucs_segment'],
        )
    except Exception as e:
        print(f'  [ERROR] {model} failed: {e}')
        results_per_model[model] = []
        continue

    # Tag output filenames with the model so they don't collide across iterations
    renamed = []
    for item in res:
        if item.status != 'done' or not item.output_path:
            continue
        p = Path(item.output_path)
        if not p.exists():
            continue
        new_p = p.with_name(f'{p.stem}-{model}{p.suffix}')
        shutil.move(str(p), str(new_p))
        item.output_path = new_p
        renamed.append(new_p)
        print(f'  Renamed to: {new_p.name}')
    results_per_model[model] = res
    print(f'  Model {model} elapsed: {time.time()-t0:.1f}s')

print(f'\n{"="*60}\nTotal elapsed: {time.time()-t0_total:.1f}s\n{"="*60}')
for model, res in results_per_model.items():
    statuses = [it.status for it in res]
    print(f'  {model:15}  →  {statuses}')
print(f'\nOutputs in: {out_dir}')

## 9. Download outputs (local mode only)

When saving to Drive, skip this step and pick them up from Drive directly.

In [ ]:
from google.colab import files
from pathlib import Path
import os

drive_mounted = os.path.ismount('/content/drive')
n_done = n_drive = n_dl = 0
for model, res in results_per_model.items():
    for it in res:
        if it.status != 'done' or not it.output_path:
            continue
        n_done += 1
        p = Path(it.output_path)
        if not p.exists():
            print(f'  [WARN] {p.name} marked done but file not found at {p}')
            continue
        on_drive = drive_mounted and str(p).startswith('/content/drive/')
        if on_drive:
            n_drive += 1
            print(f'  (skip) {p.name} — already in Drive')
            continue
        print(f'  Downloading {p.name}...')
        files.download(str(p))
        n_dl += 1
print(f'\n{n_done} done | {n_drive} in Drive | {n_dl} downloaded')

## Reset for next run

Clears cached `batch_dub` modules and removes the cloned repo so a fresh `git pull` happens on the next setup.

In [ ]:
import sys
for _m in list(sys.modules):
    if _m.startswith('batch_dub'):
        del sys.modules[_m]
print('Cleared cached batch_dub modules')

!rm -rf /content/Sta-RU
print('Removed /content/Sta-RU')

## Free disk: processing cache

In [ ]:
import shutil, os
for p in ('/tmp/sta-ru-cache', '/tmp/sta-ru-edge', '/tmp/sta-ru-work', '/tmp/dbg', '/tmp/no_voice'):
    if os.path.exists(p):
        shutil.rmtree(p, ignore_errors=True)
        print(f'Removed {p}')
    else:
        print(f'(skip) {p} not present')
!df -h /tmp | tail -1